In [2]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader
 
from lib.dataset.dataloader import AMSR2Dataset, collate_crop_to_min, collate_pad_to_max
from lib.model.Baseline import EncDec

In [ ]:
## Config — match your training setup ###
CACHE_DIR      = '/dmidata/projects/asip-cms/ninna_msc/zarr_cache'
OUTPUT_DIR     = "/dmidata/users/nili/Master/Master-thesis---Super-resolution-sea-ice-concentration-using-generative-AI/outputs/training/baseline"
CKPT_PATH      = os.path.join(OUTPUT_DIR, 'best_baseline_model_2.pth')
SAVE_PATH      = os.path.join(OUTPUT_DIR, 'prediction_with_amsr2.png')
 
BATCH_SIZE        = 8
NUM_WORKERS       = 4
SAMPLE_IDX        = 0   # which sample in the batch to plot
 
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
 
### Load dataset ###
val_dataset = AMSR2Dataset(CACHE_DIR, split='val')
val_loader  = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    collate_fn=collate_crop_to_min,
)
 
### Load checkpoint — read architecture params from saved state ###
ckpt              = torch.load(CKPT_PATH, map_location=device, weights_only=True)
AMSR2_IN_CHANNELS = ckpt["in_channels"]
FEATURES          = ckpt["features"]
print(f"Checkpoint: epoch={ckpt['epoch']}  in_channels={AMSR2_IN_CHANNELS}  features={FEATURES}  val_loss={ckpt['val_loss']:.4f}")
model = EncDec(in_channels=AMSR2_IN_CHANNELS, features=FEATURES).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()